# 09 — Le PONT vers le monde réel : reconnaître un rythme YouTube

Jusqu'ici tout reposait sur le corpus annoté. Ici, le test décisif : un système entraîné **uniquement sur le corpus** peut-il reconnaître le rythme d'une **vidéo YouTube** (mix mono, autre lieu, autres musiciens, ~20 ans plus tard) ?

Pipeline **audio-seul** (aucune annotation utilisée) :
1. **Séparer le dunun** du mix par NMF informée (dictionnaires appris sur les pistes isolées).
2. Détecter ses onsets.
3. Features d'**IOI invariantes au tempo** (intervalles entre frappes normalisés par médiane glissante — insensible à l'accelerando).
4. Classifieur entraîné sur les 15 prises du corpus **passées dans le même pipeline**.

> Pourquoi les IOI et pas le profil métrique ? Sur YouTube on n'a pas la grille des cycles. Les ratios d'intervalles, eux, ne demandent aucun alignement.

## 0. Imports, dictionnaires NMF & fonctions du pipeline

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score
import djembe_utils as dj

c = dj.list_corpus()
Wd = dj.learn_spectral_dict(c[c['code'].isin(['D1','D2'])].to_dict('records'), dj.KD)
Wj = dj.learn_spectral_dict(c[c['code'].isin(['J1','J2'])].to_dict('records'), dj.KJ)
W = np.hstack([Wd, Wj])
print('Dictionnaires spectraux :', W.shape, f'({dj.KD} dunun + {dj.KJ} jembé)')

def dunun_onsets_from_mix(mix):
    """mix mono -> onsets du dunun séparé par NMF."""
    H = dj.nmf_activations(dj.spec(mix), W)
    act = H[:dj.KD].sum(0).astype(np.float32)
    return librosa.onset.onset_detect(onset_envelope=act, sr=dj.SR, hop_length=dj.HOP,
                                      units='time', delta=0.15, wait=4)

def ioi_features(on):
    """features tempo-invariantes depuis les temps d'onsets."""
    iois = np.diff(on); iois = iois[(iois>0.05) & (iois<2.5)]
    if len(iois) < 20: return None
    med = pd.Series(iois).rolling(9, center=True, min_periods=3).median().values
    v = iois/med
    hist, _ = np.histogram(v, bins=np.linspace(0.3, 3.0, 14), density=True)
    return list(hist) + [np.std(v), np.mean(np.abs(np.log(v[1:]/v[:-1])))]

## 1. Entraîner & valider sur le corpus (même pipeline, ~2 min)
Honnêteté : les prises du corpus passent dans **le même pipeline audio** (mix reconstruit → NMF → IOI), pas par les annotations.

In [ ]:
rows = []
for (piece, take), _ in c.groupby(['piece','take']):
    mix, _codes = dj.build_mix(piece, int(take))
    f = ioi_features(dunun_onsets_from_mix(mix))
    if f: rows.append({'piece':piece, **{f'f{i}':x for i,x in enumerate(f)}})
d = pd.DataFrame(rows); F = [k for k in d.columns if k.startswith('f')]
pred = cross_val_predict(RandomForestClassifier(n_estimators=400, random_state=0),
                         d[F].values, d['piece'].values, cv=LeaveOneOut())
print('Leave-one-out (audio seul) :', round(accuracy_score(d['piece'], pred), 3))
display(pd.crosstab(d['piece'], pred))
clf = RandomForestClassifier(n_estimators=400, random_state=0).fit(d[F].values, d['piece'].values)

## 2. LE TEST : le Woloso YouTube

In [ ]:
y, _ = librosa.load('archive/youtube/Woloso.wav', sr=dj.SR, mono=True)
on = dunun_onsets_from_mix(y)
f = ioi_features(on)
proba = clf.predict_proba([f])[0]
print(len(on), 'onsets de dunun séparés du mix YouTube\n')
for p, pr in zip(clf.classes_, proba):
    print(f'  {p:10s} : {pr:.2f}', '◀' if pr==proba.max() else '')
plt.bar(clf.classes_, proba, color=['#2b6cff','#d63bd6','#2ec45f'])
plt.ylabel('probabilité'); plt.title('YouTube Woloso — quel rythme selon le modèle corpus ?'); plt.show()

## 3. Bonus exploratoire : les 6 autres vidéos
Ces rythmes (Djansa, Gede, ...) **ne sont pas dans le corpus** — le modèle ne peut donner que « le rythme du corpus le plus proche ». Purement indicatif.

In [ ]:
import glob, os
for w in sorted(glob.glob('archive/youtube/*.wav')):
    name = os.path.splitext(os.path.basename(w))[0]
    if name == 'Woloso': continue
    yv, _ = librosa.load(w, sr=dj.SR, mono=True)
    fv = ioi_features(dunun_onsets_from_mix(yv))
    if fv is None:
        print(f'{name:14s} : pas assez d\'onsets'); continue
    pr = clf.predict_proba([fv])[0]
    best = clf.classes_[np.argmax(pr)]
    print(f'{name:14s} : plus proche de {best:9s} ({pr.max():.2f})')

## Bilan
- **Le pont tient** : le Woloso YouTube est reconnu **Woloso** par un modèle qui n'a jamais entendu que le corpus de studio (2006-2007).
- Pipeline audio-seul : 80 % LOO sur corpus, transfert réussi sur un enregistrement réel.
- Limites : 3 rythmes seulement dans le corpus ; les autres vidéos n'ont pas de vérité terrain. Pour étendre : annoter (même grossièrement) d'autres rythmes, ou utiliser les vidéos comme données faiblement étiquetées (le titre = le label).